# TrackNetV3 — Fine-tuning con PadelTracker100

Pipeline para entrenar un detector de pelota específico para pádel.

**Estructura asumida en Drive** (`paddelstats_tracknet/padeltracker100/`):
```
padeltracker100/
  labels/
    2022_BCN_FinalF_1_ball.json
    2022_BCN_FinalM_1_ball.json
    ...
  2022_BCN_FinalF_1.mp4
  2022_BCN_FinalM_1.mp4
  2022_BCN_FinalF_1_sample.mp4
```

⏱️ **Tiempos estimados (T4 GPU):**
- Extracción de frames (~100k): ~30-40 min
- Conversión de anotaciones: ~2 min
- Fine-tuning 10 epochs: ~3-4h
- Inferencia 60s de vídeo: ~2 min

⚠️ Todo se guarda en Drive — si la sesión se corta, retoma desde donde se quedó.

In [ ]:
# PASO 0 — GPU + montar Drive
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else '❌ SIN GPU — activa T4 en Runtime')

from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR    = '/content/drive/MyDrive/paddelstats_tracknet'
DATASET_DIR = os.path.join(BASE_DIR, 'padeltracker100')
LABELS_DIR  = os.path.join(DATASET_DIR, 'labels')

# Directorio del dataset con la estructura que TrackNetV3 espera
# Los frames se extraen directamente aquí (sin symlinks)
TRACKNET_DATA_DIR = os.path.join(BASE_DIR, 'tracknetv3_dataset')

# Verificar que los archivos están donde esperamos
expected = [
    os.path.join(LABELS_DIR, '2022_BCN_FinalF_1_ball.json'),
    os.path.join(LABELS_DIR, '2022_BCN_FinalM_1_ball.json'),
    os.path.join(DATASET_DIR, '2022_BCN_FinalF_1.mp4'),
    os.path.join(DATASET_DIR, '2022_BCN_FinalM_1.mp4'),
]
all_ok = True
for f in expected:
    exists = os.path.exists(f)
    print(f'{'✅' if exists else '❌ NO ENCONTRADO'} {os.path.basename(f)}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n✅ Todos los archivos encontrados — continúa con el siguiente paso')
else:
    print('\n❌ Verifica que los archivos están en Drive en la ruta correcta')

In [ ]:
# PASO 0b — Descargar PadelTracker100 desde Zenodo (solo si no tienes los datos en Drive)
# Sáltate este paso si ya tienes los mp4s en paddelstats_tracknet/padeltracker100/
#
# v1: DOI 10.5281/zenodo.14653706 — padel-data-labels.zip (7.1 GB, vídeos + labels)
# v2: DOI 10.5281/zenodo.17020011 — solo labels.zip (70 MB, sin vídeos)
# Usamos v1 porque necesitamos los mp4s.

import os

BASE = '/content/drive/MyDrive/paddelstats_tracknet/padeltracker100'
os.makedirs(BASE, exist_ok=True)

ZIP_PATH = f'{BASE}/padel-data-labels.zip'
ZIP_URL  = 'https://zenodo.org/records/14653706/files/padel-data-labels.zip?download=1'

# 1. Descargar el zip (~7.1 GB, ~10-15 min con la conexión de Colab)
if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1e8:
    print(f'✅ Zip ya descargado ({os.path.getsize(ZIP_PATH)/1e9:.1f} GB)')
else:
    print('Descargando padel-data-labels.zip (~7.1 GB)...')
    !wget -q --show-progress -O "{ZIP_PATH}" "{ZIP_URL}"
    print(f'✅ Descargado ({os.path.getsize(ZIP_PATH)/1e9:.1f} GB)')

# 2. Extraer en padeltracker100/
mp4_f = os.path.join(BASE, '2022_BCN_FinalF_1.mp4')
if os.path.exists(mp4_f) and os.path.getsize(mp4_f) > 1e8:
    print('✅ Archivos ya extraídos')
else:
    print('Extrayendo zip...')
    !unzip -q "{ZIP_PATH}" -d "{BASE}"
    # El zip puede extraer en un subdirectorio — mover si hace falta
    import glob, shutil
    for mp4 in glob.glob(f'{BASE}/**/*.mp4', recursive=True):
        dst = os.path.join(BASE, os.path.basename(mp4))
        if mp4 != dst:
            shutil.move(mp4, dst)
    for json_f in glob.glob(f'{BASE}/**/*.json', recursive=True):
        labels_dir = os.path.join(BASE, 'labels')
        os.makedirs(labels_dir, exist_ok=True)
        dst = os.path.join(labels_dir, os.path.basename(json_f))
        if json_f != dst:
            shutil.move(json_f, dst)
    print('✅ Extracción completada')

# 3. Verificar
for f in ['2022_BCN_FinalF_1.mp4', '2022_BCN_FinalM_1.mp4',
          'labels/2022_BCN_FinalF_1_ball.json', 'labels/2022_BCN_FinalM_1_ball.json']:
    path = os.path.join(BASE, f)
    if os.path.exists(path):
        print(f'✅ {f} ({os.path.getsize(path)/1e6:.0f} MB)')
    else:
        print(f'❌ No encontrado: {f}')


In [ ]:
# PASO 1 — Instalar dependencias
!pip install tqdm pandas opencv-python-headless gdown --quiet

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Sin GPU — el entrenamiento tardará horas. Activa T4 en Runtime > Change runtime type')

In [ ]:
# PASO 2 — Explorar el JSON de anotaciones para confirmar el formato
# EJECUTA ESTO ANTES DE CONTINUAR y verifica que la salida tiene sentido
import json, os
from pathlib import Path

ball_json_f = os.path.join(LABELS_DIR, '2022_BCN_FinalF_1_ball.json')

with open(ball_json_f) as f:
    data = json.load(f)

print('Claves del JSON:', list(data.keys()))
print(f'\nNúmero de imágenes: {len(data.get("images", []))}')
print(f'Número de anotaciones: {len(data.get("annotations", []))}')

if data.get('images'):
    print(f'\nEjemplo imagen[0]: {data["images"][0]}')
    print(f'Ejemplo imagen[1]: {data["images"][1]}')

if data.get('annotations'):
    print(f'\nEjemplo anotación[0]: {data["annotations"][0]}')
    print(f'Ejemplo anotación[1]: {data["annotations"][1]}')

if data.get('categories'):
    print(f'\nCategorías: {data["categories"]}')

In [ ]:
# PASO 3 — Extraer frames directamente en la estructura de TrackNetV3
#
# Estructura destino:
#   tracknetv3_dataset/train/match1/frame/2022_BCN_FinalF_1/000000.jpg
#   tracknetv3_dataset/train/match2/frame/2022_BCN_FinalM_1/000000.jpg
#
# Resolución: 960x540 (mitad de 1080p) — TrackNetV3 redimensiona internamente a 512x288
#
# NOTA DRIVE: os.sync() cada 500 frames para forzar flush del FUSE mount.
# Si la sesión se corta, los frames ya escritos no se vuelven a extraer.

import cv2, os, time
from pathlib import Path
from tqdm import tqdm

TARGET_W, TARGET_H = 960, 540

MATCHES = [
    ('match1', '2022_BCN_FinalF_1', os.path.join(DATASET_DIR, '2022_BCN_FinalF_1.mp4')),
    ('match2', '2022_BCN_FinalM_1', os.path.join(DATASET_DIR, '2022_BCN_FinalM_1.mp4')),
]

SYNC_EVERY = 500  # Forzar flush a Drive cada N frames

for match_name, video_name, video_path in MATCHES:
    out_dir = os.path.join(TRACKNET_DATA_DIR, 'train', match_name, 'frame', video_name)
    os.makedirs(out_dir, exist_ok=True)

    # Detectar último frame extraído (reanudación)
    existing = sorted(Path(out_dir).glob('*.jpg'))
    last_idx = int(existing[-1].stem) if existing else -1

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if last_idx >= total - 10:
        cap.release()
        print(f'✅ {match_name} ({video_name}): {len(existing)} frames ya extraídos, saltando')
        continue

    print(f'Extrayendo {match_name} — {video_name}')
    print(f'  Total: {total} frames | Ya extraídos: {last_idx + 1} | Quedan: {total - last_idx - 1}')

    frame_idx = 0
    written = 0
    with tqdm(total=total, initial=last_idx + 1) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx > last_idx:
                out_path = os.path.join(out_dir, f'{frame_idx:06d}.jpg')
                resized = cv2.resize(frame, (TARGET_W, TARGET_H))
                cv2.imwrite(out_path, resized, [cv2.IMWRITE_JPEG_QUALITY, 90])
                written += 1
                # Forzar flush periódico para que Drive persista los archivos
                if written % SYNC_EVERY == 0:
                    os.sync()
            frame_idx += 1
            pbar.update(1)

    cap.release()
    os.sync()  # Flush final

    # Verificar cuántos frames existen realmente en Drive
    time.sleep(2)
    actual = len(list(Path(out_dir).glob('*.jpg')))
    print(f'✅ {match_name}: {actual} frames confirmados en Drive (esperados: {total})')
    if actual < total - 10:
        print(f'   ⚠️ Faltan {total - actual} frames — vuelve a ejecutar esta celda para reanudar')


In [ ]:
# PASO 4 — Convertir anotaciones COCO JSON → CSV formato TrackNetV3
#
# Formato TrackNetV3: Frame,Visibility,X,Y
#   Frame       — índice de frame (0-based)
#   Visibility  — 1 si pelota visible, 0 si no
#   X, Y        — centro de la pelota en píxeles (escala 960x540)
#
# category_id == 1 → Ball (hay también Wall, shot-event, etc. — ignorados)
# Los CSVs van a: tracknetv3_dataset/train/match1/csv/2022_BCN_FinalF_1_ball.csv

import json, os, time
import pandas as pd
from pathlib import Path

TARGET_W, TARGET_H = 960, 540  # Por si este paso se ejecuta sin pasar por paso 3
ORIG_W, ORIG_H = 1920, 1080    # Resolución original de los vídeos WPT 2022
BALL_CATEGORY_ID = 1            # Solo anotar la pelota, ignorar Wall y eventos de golpe

def convert_ball_json(json_path, output_csv):
    with open(json_path) as f:
        data = json.load(f)

    # Mapa image_id → número de frame
    # file_name = 'frame_000000.PNG' → frame 0
    img_map = {}
    for img in data.get('images', []):
        stem = Path(img['file_name']).stem          # 'frame_000000'
        frame_num = int(''.join(filter(str.isdigit, stem)))  # 0
        img_map[img['id']] = {
            'frame': frame_num,
            'orig_w': img.get('width', ORIG_W),
            'orig_h': img.get('height', ORIG_H),
        }

    # Anotaciones → {image_id: (cx, cy)} en resolución 960x540
    # Solo category_id == 1 (Ball). Wall y shot-events se ignoran.
    ball_anns = {}
    for ann in data.get('annotations', []):
        if ann.get('category_id') != BALL_CATEGORY_ID:
            continue
        img_id = ann['image_id']
        bbox = ann.get('bbox')  # [x, y, w, h] — formato COCO, coords float
        if bbox and img_id in img_map:
            info = img_map[img_id]
            sx = TARGET_W / info['orig_w']
            sy = TARGET_H / info['orig_h']
            cx = (bbox[0] + bbox[2] / 2) * sx
            cy = (bbox[1] + bbox[3] / 2) * sy
            ball_anns[img_id] = (round(cx), round(cy))

    # Un row por frame, ordenado por índice de frame
    rows = []
    for img_id, info in sorted(img_map.items(), key=lambda x: x[1]['frame']):
        if img_id in ball_anns:
            cx, cy = ball_anns[img_id]
            rows.append({'Frame': info['frame'], 'Visibility': 1, 'X': cx, 'Y': cy})
        else:
            rows.append({'Frame': info['frame'], 'Visibility': 0, 'X': 0, 'Y': 0})

    df = pd.DataFrame(rows)

    # Escribir en /tmp primero (filesystem local, sin FUSE) y luego copiar a Drive
    # Esto evita el problema de buffering del mount de Google Drive
    tmp_csv = f'/tmp/{Path(output_csv).name}'
    df.to_csv(tmp_csv, index=False)

    # Copiar explícitamente a Drive
    import shutil
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    shutil.copy2(tmp_csv, output_csv)

    # Forzar flush del sistema de archivos
    os.sync()

    visible = (df['Visibility'] == 1).sum()
    return len(df), visible, df

MATCH_CONFIG = [
    ('match1', '2022_BCN_FinalF_1', os.path.join(LABELS_DIR, '2022_BCN_FinalF_1_ball.json')),
    ('match2', '2022_BCN_FinalM_1', os.path.join(LABELS_DIR, '2022_BCN_FinalM_1_ball.json')),
]

total_frames_all = 0
total_vis_all = 0
first_df = None

for match_name, video_name, json_path in MATCH_CONFIG:
    csv_dir = os.path.join(TRACKNET_DATA_DIR, 'train', match_name, 'csv')
    os.makedirs(csv_dir, exist_ok=True)
    out_csv = os.path.join(csv_dir, f'{video_name}_ball.csv')

    n_frames, n_vis, df = convert_ball_json(json_path, out_csv)
    if first_df is None:
        first_df = df

    total_frames_all += n_frames
    total_vis_all += n_vis
    rate = n_vis / n_frames * 100 if n_frames > 0 else 0

    # Verificar que el archivo realmente existe y tiene el tamaño correcto
    time.sleep(1)  # Dar tiempo al FUSE para completar la escritura
    if os.path.exists(out_csv):
        size_kb = os.path.getsize(out_csv) / 1024
        print(f'✅ {match_name} ({video_name}): {n_frames} frames, {n_vis} con pelota ({rate:.1f}%)')
        print(f'   Archivo: {out_csv} ({size_kb:.0f} KB)')
    else:
        print(f'❌ {match_name}: el archivo NO se guardó en Drive — {out_csv}')

total_rate = total_vis_all / total_frames_all * 100 if total_frames_all > 0 else 0
print(f'\nTotal: {total_frames_all} frames, {total_vis_all} con pelota ({total_rate:.1f}%)')

# Mostrar muestra del CSV para verificar que las coordenadas tienen sentido
if first_df is not None:
    print('\nPrimeras filas con pelota visible (Visibility=1):')
    print(first_df[first_df['Visibility'] == 1].head(5).to_string())
    print('\nPrimeras filas sin pelota (Visibility=0):')
    print(first_df[first_df['Visibility'] == 0].head(3).to_string())


In [ ]:
# PASO 5 — Verificar estructura del dataset + crear test y val si no existen

import os, shutil, pandas as pd
from pathlib import Path

print('Verificando estructura del dataset...')
all_ok = True

for match_name, video_name, _ in MATCH_CONFIG:
    csv_dir   = os.path.join(TRACKNET_DATA_DIR, 'train', match_name, 'csv')
    frame_dir = os.path.join(TRACKNET_DATA_DIR, 'train', match_name, 'frame', video_name)

    csv_files   = list(Path(csv_dir).glob('*.csv')) if os.path.exists(csv_dir) else []
    frame_files = list(Path(frame_dir).glob('*.jpg')) if os.path.exists(frame_dir) else []

    print(f'{match_name}:')
    print(f'  {"✅" if csv_files else "❌"} CSVs: {len(csv_files)} archivos')
    print(f'  {"✅" if frame_files else "❌"} Frames: {len(frame_files)} imágenes')

    if not csv_files or not frame_files:
        all_ok = False

def create_split(split, src_match, src_video, n_frames):
    """Crea un split (test/val) con los últimos n_frames del match indicado."""
    dst_frame_dir = os.path.join(TRACKNET_DATA_DIR, split, 'match1', 'frame', src_video)
    dst_csv_dir   = os.path.join(TRACKNET_DATA_DIR, split, 'match1', 'csv')

    if os.path.exists(dst_frame_dir) and len(list(Path(dst_frame_dir).glob('*.jpg'))) > 0:
        n = len(list(Path(dst_frame_dir).glob('*.jpg')))
        print(f'✅ {split}/match1 ya existe ({n} frames)')
        return

    os.makedirs(dst_frame_dir, exist_ok=True)
    os.makedirs(dst_csv_dir, exist_ok=True)

    src_frames = sorted(Path(os.path.join(TRACKNET_DATA_DIR, 'train', src_match, 'frame', src_video)).glob('*.jpg'))[-n_frames:]
    print(f'Creando {split}/match1 ({len(src_frames)} frames)...')
    for f in src_frames:
        dst = Path(dst_frame_dir) / f.name
        if not dst.exists():
            shutil.copy2(f, dst)

    existing = sorted([int(f.stem) for f in Path(dst_frame_dir).glob('*.jpg')])
    src_csv = os.path.join(TRACKNET_DATA_DIR, 'train', src_match, 'csv', f'{src_video}_ball.csv')
    df = pd.read_csv(src_csv)
    df[df['Frame'].isin(existing)].to_csv(
        os.path.join(dst_csv_dir, f'{src_video}_ball.csv'), index=False
    )
    print(f'✅ {split}/match1 creado ({len(existing)} frames)')

# Test: últimos 300 frames de match1
create_split('test', 'match1', '2022_BCN_FinalF_1', 300)
# Val: últimos 500 frames de match2
create_split('val', 'match2', '2022_BCN_FinalM_1', 500)

if all_ok:
    print('\n✅ Estructura correcta — listo para el paso 6')
else:
    print('\n❌ Faltan archivos — ejecuta los pasos anteriores primero')


In [ ]:
# PASO 6 — Clonar TrackNetV3 + descargar pesos base (bádminton)
import gdown, os, shutil, torch
from pathlib import Path

TRACKNET_DIR = os.path.join(BASE_DIR, 'TrackNetV3')
CKPT_DIR     = os.path.join(BASE_DIR, 'ckpts')
EXP_DIR      = os.path.join(BASE_DIR, 'exp_padel')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)

# Clonar TrackNetV3
if not os.path.exists(os.path.join(TRACKNET_DIR, 'train.py')):
    !git clone https://github.com/qaz812345/TrackNetV3.git "{TRACKNET_DIR}" --quiet
    print('✅ TrackNetV3 clonado')
else:
    print('✅ TrackNetV3 ya existe')

# --- Parches necesarios para nuestro dataset ---

# 1. dataset.py: data_dir apunta a Drive en vez de 'data' relativo
dataset_py = os.path.join(TRACKNET_DIR, 'dataset.py')
with open(dataset_py) as f:
    src = f.read()
if "data_dir = 'data'" in src:
    src = src.replace("data_dir = 'data'", f"data_dir = '{TRACKNET_DATA_DIR}'")
    src = src.replace(
        "w, h = Image.open(os.path.join(rally_dir, f'0.{IMG_FORMAT}')).size",
        "first = sorted([x for x in os.listdir(rally_dir) if x.endswith(IMG_FORMAT)])[0]; w, h = Image.open(os.path.join(rally_dir, first)).size"
    )
    src = src.replace(
        "f_file = np.array([os.path.join(rally_dir, f'{f_id}.{IMG_FORMAT}') for f_id in label_df['Frame']])",
        "f_file = np.array([os.path.join(rally_dir, f'{int(f_id):06d}.{IMG_FORMAT}') for f_id in label_df['Frame']])"
    )
    with open(dataset_py, 'w') as f:
        f.write(src)
    print('✅ dataset.py parcheado (data_dir + zero-padding)')
else:
    print('✅ dataset.py ya parcheado')

# 2. utils/general.py: IMG_FORMAT jpg en vez de png
general_py = os.path.join(TRACKNET_DIR, 'utils', 'general.py')
with open(general_py) as f:
    src = f.read()
if "IMG_FORMAT = 'png'" in src:
    src = src.replace("IMG_FORMAT = 'png'", "IMG_FORMAT = 'jpg'")
    with open(general_py, 'w') as f:
        f.write(src)
    print('✅ utils/general.py parcheado (IMG_FORMAT → jpg)')
else:
    print('✅ utils/general.py ya parcheado')

# --- Descargar pesos pre-entrenados (bádminton) ---
tracknet_pt = os.path.join(CKPT_DIR, 'TrackNet_best.pt')
if not os.path.exists(tracknet_pt):
    print('Descargando pesos pre-entrenados (bádminton)...')
    ckpt_zip = os.path.join(CKPT_DIR, 'ckpts.zip')
    gdown.download(id='1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA', output=ckpt_zip, quiet=False)
    !unzip -q "{ckpt_zip}" -d "{CKPT_DIR}"
    for pt in Path(CKPT_DIR).rglob('*.pt'):
        dst = os.path.join(CKPT_DIR, pt.name)
        if str(pt) != dst:
            shutil.move(str(pt), dst)
    print('✅ Pesos descargados')
else:
    print('✅ Pesos ya descargados')

for pt_name in ['TrackNet_best.pt', 'InpaintNet_best.pt']:
    src_pt = os.path.join(CKPT_DIR, pt_name)
    dst_pt = os.path.join(EXP_DIR, pt_name)
    if os.path.exists(src_pt) and not os.path.exists(dst_pt):
        shutil.copy(src_pt, dst_pt)
        print(f'Copiado {pt_name} → exp_padel/')

# Preparar TrackNet_cur.pt para --resume_training
cur_pt = os.path.join(EXP_DIR, 'TrackNet_cur.pt')
if not os.path.exists(cur_pt):
    shutil.copy(os.path.join(EXP_DIR, 'TrackNet_best.pt'), cur_pt)

# Parchear checkpoint: añadir claves que el código nuevo espera
# - param_dict: hiperparámetros restaurados por ResumeArgumentParser
# - nivel raíz: estado de entrenamiento leído directamente por train.py
ckpt = torch.load(cur_pt, map_location='cpu')
updated = False
param_defaults = {'mask_ratio': 0, 'max_val_acc': 0}
for k, v in param_defaults.items():
    if k not in ckpt.get('param_dict', {}):
        ckpt.setdefault('param_dict', {})[k] = v
        updated = True
root_defaults = {'max_val_acc': 0}
for k, v in root_defaults.items():
    if k not in ckpt:
        ckpt[k] = v
        updated = True
if updated:
    torch.save(ckpt, cur_pt)
    print('✅ Checkpoint parcheado (claves faltantes añadidas)')
else:
    print('✅ Checkpoint ya al día')

print('\nInstalando dependencias de TrackNetV3...')
!pip install parse --quiet
!pip install -r "{TRACKNET_DIR}/requirements.txt" --quiet 2>&1 | tail -3
print('✅ Listo')


In [ ]:
# PASO 7 — Preprocesar dataset (genera backgrounds por median filtering)
# TrackNetV3 genera una imagen de 'background' por segmento para eliminar
# elementos estáticos (pista, red) y destacar la pelota.
# Tiempo estimado: ~10-15 min en T4

import sys, os

# 'parse' no viene en Colab por defecto pero lo necesita preprocess.py
!pip install parse --quiet

sys.path.insert(0, TRACKNET_DIR)
os.chdir(TRACKNET_DIR)

print('Preprocesando dataset...')
print('Genera background frames por median filtering — ~10-15 min')
print()

!python preprocess.py \
    --data_dir "{TRACKNET_DATA_DIR}" \
    --split_ratio 0.9

print('\n✅ Preprocesamiento completado')


In [ ]:
# PASO 8 — Fine-tuning de TrackNet
# data_dir está hardcodeado en dataset.py (parcheado en paso 6)
# train.py NO acepta --data_dir — lo lee de dataset.py directamente

import os
EPOCHS = 10

os.chdir(TRACKNET_DIR)

print(f'Iniciando fine-tuning — {EPOCHS} epochs')
print(f'Directorio experimento: {EXP_DIR}')
print(f'Si la sesión se corta, vuelve a ejecutar esta celda — retomará automáticamente.')
print()

!python train.py \
    --model_name TrackNet \
    --epochs {EPOCHS} \
    --save_dir "{EXP_DIR}" \
    --resume_training \
    --verbose


In [ ]:
# PASO 8b — Ver progreso del entrenamiento (ejecuta en cualquier momento)
import pandas as pd
from pathlib import Path

print('Checkpoints guardados en exp_padel/:')
pts = sorted(Path(EXP_DIR).glob('*.pt'))
if pts:
    for pt in pts:
        print(f'  {pt.name} ({pt.stat().st_size/1e6:.1f} MB)')
else:
    print('  (ninguno todavía)')

# Leer log de entrenamiento
log_files = list(Path(EXP_DIR).rglob('*.csv'))
if log_files:
    log = pd.read_csv(log_files[0])
    print(f'\nEpochs completados: {len(log)}')
    if len(log) > 0:
        print(log.to_string())
        import matplotlib.pyplot as plt
        if 'loss' in log.columns:
            plt.plot(log['loss'], label='train loss')
            if 'val_loss' in log.columns:
                plt.plot(log['val_loss'], label='val loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            plt.title('TrackNetV3 fine-tuning — pádel')
            plt.show()
else:
    print('\nNo hay log de entrenamiento todavía.')

In [ ]:
# PASO 9 — Inferencia sobre tu vídeo de pádel
# Sube el vídeo de prueba (ej: test_60s.mp4)

import os
from google.colab import files

print('Sube tu vídeo de prueba (ej: test_60s.mp4):')
uploaded = files.upload()
VIDEO_PATH = '/content/' + list(uploaded.keys())[0]
print(f'✅ Vídeo: {VIDEO_PATH}')

PRED_DIR = os.path.join(BASE_DIR, 'predictions')
os.makedirs(PRED_DIR, exist_ok=True)

TRACKNET_FT = os.path.join(EXP_DIR, 'TrackNet_best.pt')   # Fine-tuned en pádel
INPAINT_PT  = os.path.join(CKPT_DIR, 'InpaintNet_best.pt') # Sin fine-tuning (no tenemos datos de inpaint)

print(f'\nUsando modelo fine-tuned: {TRACKNET_FT}')
print('Corriendo inferencia...')

!python "{TRACKNET_DIR}/predict.py" \
    --video_file "{VIDEO_PATH}" \
    --tracknet_file "{TRACKNET_FT}" \
    --inpaintnet_file "{INPAINT_PT}" \
    --save_dir "{PRED_DIR}" \
    --output_video \
    --large_video

In [ ]:
# PASO 10 — Resultados + descarga
import pandas as pd
from pathlib import Path
from google.colab import files

pred_csvs = list(Path(PRED_DIR).glob('*.csv'))
if pred_csvs:
    df = pd.read_csv(pred_csvs[0])
    visible = df[df['Visibility'] == 1]
    det_rate = len(visible) / len(df) * 100

    print('=' * 50)
    print('RESULTADO FINAL')
    print('=' * 50)
    print(f'Frames procesados : {len(df)}')
    print(f'Pelota detectada  : {len(visible)}')
    print(f'Detection rate    : {det_rate:.1f}%')
    print()
    print('Comparativa:')
    print(f'  YOLOv8 single-frame (baseline) : ~20%')
    print(f'  WASB/TrackNetV2 sin fine-tuning :   0%')
    print(f'  TrackNetV3 fine-tuned (pádel)   : {det_rate:.1f}%  ← ESTE')
    print()
    if det_rate > 60:
        print('✅ Listo para integrar en el pipeline de PaddelStats')
    elif det_rate > 30:
        print('⚠️ Mejora real pero insuficiente — prueba más epochs (modifica EPOCHS=20 en paso 8)')
    else:
        print('❌ Poca mejora — revisa el paso 2 (formato JSON) y el paso 4 (conversión CSV)')
else:
    print('No se encontraron CSVs en', PRED_DIR)

# Descargar vídeo anotado
pred_videos = list(Path(PRED_DIR).glob('*.mp4'))
if pred_videos:
    print(f'\nDescargando vídeo anotado: {pred_videos[0].name}')
    files.download(str(pred_videos[0]))
else:
    print('No se encontró vídeo de salida en', PRED_DIR)